### Load data 

In [1]:
# Load CSEC2017 synthetic + KD2017 datasets 
from datasets import load_from_disk
import datasets
datasets.disable_caching()
import sys
sys.path.append("/nobackup/proj/disk/naiss2024-22-903/personal/LLMedu") 

num_classes = 9 
KAs = {"0": "miscellaneous (this includes Computer Science, Business and Law, Communication and Networking, Information Technology, Cyberspace Practice, Pedagogy, and Intelligence)",
           "1": "data security", 
           "2": "software security",
           "3": "component security", 
           "4": "connection security", 
           "5": "system security", 
           "6": "human security", 
           "7": "organizational security",
           "8": "societal security"}

# edited after IEEE Access reviewer comments 
KAs = {
    0: {
        "name": "Miscellaneous",
        "definition": "Computer Science, Business & Law, Communication & Networking, Information Technology, Cyberspace Practice, Pedagogy, Intelligence"
    },
    1: {
        "name": "Data Security",
        "definition": "Basic cryptography concepts, Digital forensics, End-to-end secure communications, Data integrity & authentication, Information storage security"
    },
    2: {
        "name": "Software Security",
        "definition": "Fundamental design principles, Security requirements, Implementation issues, Static & dynamic testing, Configuring & patching, Ethics"
    },
    3: {
        "name": "Component Security",
        "definition": "Vulnerabilities of system components, Component lifecycle, Secure component design principles, Supply chain management security, Security testing, Reverse engineering"
    },
    4: {
        "name": "Connection Security",
        "definition": "Systems, architecture, models, standards, Physical component interfaces, Software component interfaces, Connection attacks, Transmission attacks"
    },
    5: {
        "name": "System Security",
        "definition": "Holistic approach, Security policy, Authentication, Access control, Monitoring, Recovery, Testing, Documentation"
    },
    6: {
        "name": "Human Security",
        "definition": "Identity management, Social engineering, Awareness & understanding, Social behavioral privacy & security, Personal data privacy & security"
    },
    7: {
        "name": "Organizational Security",
        "definition": "Risk management, Governance & policy, Laws, ethics, & compliance, Strategy & planning"
    },
    8: {
        "name": "Societal Security",
        "definition": "Cybercrime, Cyber law, Cyber ethics, Cyber policy, Privacy"
    }
}

from utils.load_data import preprocess
# KDs 
KD_dataset = datasets.load_dataset("csv",data_files={"train": "/nobackup/proj/disk/naiss2024-22-903/personal/LLMedu/data/train_data.csv"}, split='train')
KD_dataset = KD_dataset.remove_columns('KSAT ID')
KD_dataset = KD_dataset.map(preprocess)
print(KD_dataset)

from utils.load_data import preprocess_csec, preprocess_csec8
# CSEC2017 specific! CHANGED: train_CSEC2017b to train_CSEC2017c to include class 0 
csec_dataset = datasets.load_dataset("csv",data_files={"train": "/nobackup/proj/disk/naiss2024-22-903/personal/LLMedu/data/train_CSEC2017c.csv"}, split='train')
csec_dataset = csec_dataset.remove_columns('Statement Description')
csec_dataset = csec_dataset.remove_columns('label')
csec_dataset = csec_dataset.select_columns(['topics','0','1','2','3','4','5','6','7','8'])
csec_dataset = csec_dataset.map(preprocess_csec)
print(csec_dataset)

Map:   0%|          | 0/576 [00:00<?, ? examples/s]

Dataset({
    features: ['0', '1', '2', '3', '4', '5', '6', '7', '8', 'Statement Description', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 576
})


Map:   0%|          | 0/2143 [00:00<?, ? examples/s]

Dataset({
    features: ['topics', '0', '1', '2', '3', '4', '5', '6', '7', '8', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 2143
})


In [2]:
import json
from tqdm import tqdm
import re 

def label_KD_prompt(knowledge, KAs):
        """
        Formats a single question+answers into a list of message dictionaries for the pipeline.
        """
        options_str = ', '.join([f"{key}) {value}" for key, value in KAs.items()])
        instructions = (
            "You are a helpful AI assistant.\n"
            "Instructions:\n"
            "a. Carefully read the knowledge statement.\n"
            "b. Choose one or more of the following (0, 1, 2, 3, 4, 5, 6, 7, 8).\n"
            "c. Do NOT include any explanation or additional text in the response.\n"
           # "d. Always return the answer in this format: 'answer'. "
            #"For example, if the correct answers are 0 and 1, then return 0,1.\n\n"
        )
    
        messages = [
            {"role": "system", "content": instructions},
            {"role": "user", "content": 
            f"#Question: Classify the following statement {knowledge} into one or multiple of the following knowledge areas: \nOptions: {options_str}"}
        ]
        return messages

In [3]:
def process_label(label): 
    output = [int(s) for s in re.findall(r'\d+', label)]
    output = str(output)
    output = output.replace(' ','')[1:-1]
    output = [int(num) for num in output.split(',')]
    return output 

In [4]:
# clean datasets and convert to pandas 
import pandas as pd 
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from utils.load_data import clean_text

feature_type = 'tf-idf' 

pandas_dataCSEC = pd.DataFrame({'topics': csec_dataset['topics'], 'labels': csec_dataset['labels']})
pandas_dataCSEC['cleaned'] = pandas_dataCSEC['topics'].apply(clean_text)

# remove knowledge of (indiscriminative)
def remove_knowledge_of(example):
    example = example[13:]
    return example

pandas_dataKD = pd.DataFrame({'topics': KD_dataset['Statement Description'], 'labels': KD_dataset['labels']})
pandas_dataKD['cleaned'] = pandas_dataKD['topics'].apply(clean_text).apply(remove_knowledge_of)
# merge fine-tuning datasets 
mergeds = pd.concat([pd.DataFrame({'topics': pandas_dataCSEC['cleaned'], 'labels': pandas_dataCSEC['labels']}), 
                     pd.DataFrame({'topics': pandas_dataKD['cleaned'], 'labels': pandas_dataKD['labels']})]).reset_index() # for
# convert labels to int 
mergeds['labels'] = mergeds['labels'].apply(lambda x: [int(i) for i in x])

In [5]:
# DeepSeek
from openai import OpenAI
# for DeepSeek-V3, set model='deepseek-chat' 
# for DeepSeek-R1, set model='deepseek-reasoner' 

client = OpenAI(api_key ="sk-b14645cb792542729d8ac6b60626fef8", base_url="https://api.deepseek.com")
response = client.chat.completions.create(
    model="deepseek-chat",
    messages = [{"role": "system", "content": "You are a helpful assistant"}, 
                {"role": "user", "content": "Hello"}, 

               ], 
    stream = False)

print(response.choices[0].message.content)

Hello! I'm here to help you with anything you need. How can I assist you today?


In [6]:
import numpy as np
num_classes = 9 
KD = mergeds['topics'][2]
query = label_KD_prompt(KD, KAs)
print(query)
response = client.chat.completions.create(
    model="deepseek-chat",
    messages = query, 
    stream = False)
print(response.choices[0].message.content)

[{'role': 'system', 'content': 'You are a helpful AI assistant.\nInstructions:\na. Carefully read the knowledge statement.\nb. Choose one or more of the following (0, 1, 2, 3, 4, 5, 6, 7, 8).\nc. Do NOT include any explanation or additional text in the response.\n'}, {'role': 'user', 'content': "#Question: Classify the following statement data integrity into one or multiple of the following knowledge areas: \nOptions: 0) {'name': 'Miscellaneous', 'definition': 'Computer Science, Business & Law, Communication & Networking, Information Technology, Cyberspace Practice, Pedagogy, Intelligence'}, 1) {'name': 'Data Security', 'definition': 'Basic cryptography concepts, Digital forensics, End-to-end secure communications, Data integrity & authentication, Information storage security'}, 2) {'name': 'Software Security', 'definition': 'Fundamental design principles, Security requirements, Implementation issues, Static & dynamic testing, Configuring & patching, Ethics'}, 3) {'name': 'Component Se

In [7]:
def Zero_Shot_sample(KD): 
    query = label_KD_prompt(KD, KAs)
    predicted_label = process_label(submit_message_LLM(model2, query))
    #print('actual label: ',mergeds['labels'][1])
    one_hot =[1 if j in predicted_label else 0 for j in range(num_classes)]
    #print('predicted label: ',one_hot)
    return one_hot

def Zero_Shot_sample(KD, deepseek=True): 
    query = label_KD_prompt(KD, KAs)
    if deepseek == True: 
        model = "deepseek-chat" 
    else: 
        model = "gpt-4o"
    response = client.chat.completions.create(
    model=model,
    messages = query, 
    stream = False)
    predicted_label = process_label(response.choices[0].message.content)
    one_hot =[1 if j in predicted_label else 0 for j in range(num_classes)]
    return one_hot 

In [8]:
### Inserted after IEEE Reviewer comments 
#1. Load independent test set 
import pandas as pd 
test_df = pd.read_excel('/nobackup/proj/disk/naiss2024-22-903/personal/LLMedu/data/Topics_Annotation.xlsx')
print(test_df.head())
# label the test set using DeepSeek
pred = test_df['Topic'].apply(Zero_Shot_sample)
print(pred)

                                Topic Annotator 1 Annotator 2  Annotator 3   \
0                     Ethical Hacking         1,6          6,8        2,5,6   
1  Network and vulnerability scanning           4      3,4,5,7          1,4   
2       Exploit development platforms         2,7            3          2,3   
3                 Command and control           0            2        2,3,6   
4                   Password cracking           1          5,6        1,2,3   

               Sara  Valtteri         Paul CuricuLLM (one specific run)  
0  0,1,2,3,4,5,6,7,8    2,7,8  3,2,4,5,6,0                            8  
1                4,5        5          4,0                            4  
2                  2        2        2,3,4                            2  
3                  0        7      0,2,4,5                            0  
4                0,1        5        1,2,6                            6  
0     [0, 0, 1, 0, 0, 0, 0, 0, 0]
1     [0, 0, 0, 0, 1, 0, 0, 0, 0]
2     [0, 0, 

In [9]:
pred.to_csv("output/pred_ZS_DeepSeek.csv", index=False) 
pred2 = pd.read_csv("output/pred_ZS_DeepSeek.csv")
pred2.head()

,Topic
0,"[0, 0, 1, 0, 0, 0, 0, 0, 0]"
1,"[0, 0, 0, 0, 1, 0, 0, 0, 0]"
2,"[0, 0, 1, 0, 0, 0, 0, 0, 0]"
3,"[1, 0, 0, 0, 0, 0, 0, 0, 0]"
4,"[0, 1, 0, 0, 0, 0, 0, 0, 0]"


In [11]:
# DeepSeek 
# To do: Train 5 times and average the resulting metrics. 
# To do: 5 different random seeds as well. 
from sklearn.model_selection import KFold, RepeatedKFold
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score 
n_repeats = 1 # number of seeds 
n_splits = 10 # number of k splits 
precisions = []
recalls = [] 
f1s = [] 
accs = []
for _ in range(n_repeats): 
    kf = KFold(n_splits=n_splits,  shuffle=True) 
    precision = 0 
    recall = 0 
    f1 = 0 
    acc = 0 
    for i, (train, test) in enumerate(kf.split(mergeds)):  # split into k-folds
        print('split ',i)
        # extract train and test dataset for k-folds cross-validation
        train_ds = mergeds.loc[train] 
        test_ds = mergeds.loc[test] 
        # do zero-shot labelling on the test dataset 
        predicted_labels = test_ds['topics'].apply(Zero_Shot_sample)
        # compute metrics on the validation dataset 
        report = classification_report(test_ds['labels'].tolist(),predicted_labels.tolist()
                                       ,output_dict=True, zero_division=0)
        accuracy = accuracy_score(test_ds['labels'].tolist(), predicted_labels.tolist())
        # gather and normalize metrics by the length of the test dataset 
        acc += (accuracy * len(test_ds)) / len(mergeds)
        precision += (report['macro avg']['precision'] * len(test_ds)) / len(mergeds)
        recall += (report['macro avg']['recall'] * len(test_ds)) / len(mergeds)
        f1 += (report['macro avg']['f1-score'] * len(test_ds)) / len(mergeds)
        print(report['macro avg'])
        # Inside your fold loop, after getting the report:
        print("Per-class metrics:")
        for class_name, metrics in report.items():
            if class_name not in ['macro avg', 'weighted avg', 'accuracy']:
                print(f"  {class_name}: P={metrics['precision']:.3f}, R={metrics['recall']:.3f}, F1={metrics['f1-score']:.3f}")
        
        print(f"Macro avg: P={report['macro avg']['precision']:.3f}, R={report['macro avg']['recall']:.3f}, F1={report['macro avg']['f1-score']:.3f}")

    precisions.append(precision) 
    recalls.append(recall) 
    f1s.append(f1)
    accs.append(acc) 
    #break 
# print the results
print('precision: ',np.mean(precisions))
print('precision std: ',np.std(precisions))
print('recall: ',np.mean(recalls))
print('recall std: ',np.std(recalls))
print('f1: ',np.mean(f1s))
print('f1 std: ',np.std(f1s))
print('accuracy: ',np.mean(accs))
print('accuracy std: ',np.std(accs))

split  0
{'precision': 0.4071297259145159, 'recall': 0.3031819116683488, 'f1-score': 0.3340997944745554, 'support': 371.0}
Per-class metrics:
  0: P=0.148, R=0.105, F1=0.123
  1: P=0.667, R=0.314, F1=0.427
  2: P=0.579, R=0.282, F1=0.379
  3: P=0.278, R=0.227, F1=0.250
  4: P=0.421, R=0.400, F1=0.410
  5: P=0.139, R=0.323, F1=0.194
  6: P=0.483, R=0.438, F1=0.459
  7: P=0.604, R=0.390, F1=0.474
  8: P=0.346, R=0.250, F1=0.290
  micro avg: P=0.382, R=0.315, F1=0.346
  samples avg: P=0.395, R=0.330, F1=0.344
Macro avg: P=0.407, R=0.303, F1=0.334
split  1
{'precision': 0.4403462196179023, 'recall': 0.312602749219732, 'f1-score': 0.35586642026789317, 'support': 396.0}
Per-class metrics:
  0: P=0.375, R=0.180, F1=0.243
  1: P=0.500, R=0.250, F1=0.333
  2: P=0.700, R=0.389, F1=0.500
  3: P=0.263, R=0.185, F1=0.217
  4: P=0.500, R=0.393, F1=0.440
  5: P=0.211, R=0.375, F1=0.270
  6: P=0.562, R=0.439, F1=0.493
  7: P=0.518, R=0.358, F1=0.423
  8: P=0.333, R=0.244, F1=0.282
  micro avg: P=0.412

In [12]:
np.savetxt('output/precision_ZS_DeepSeek.txt', precisions, delimiter=',')
np.savetxt('output/recall_ZS_DeepSeek.txt', recalls, delimiter=',')
np.savetxt('output/f1_ZS_DeepSeek.txt', f1s, delimiter=',')
np.savetxt('output/accuracy_ZS_DeepSeek.txt', accs, delimiter=',')

In [11]:
import numpy as np 
print(np.mean(precisions))
print(np.std(precisions))
print(np.mean(recalls))
print(np.std(recalls))
print(np.mean(f1s))
print(np.std(f1s))
print(np.mean(accuracy))
print(np.std(accuracy))

0.3774178117695753
0.0015001368374326467
0.48521111599919226
0.001024204981642221
0.40561114056818887
0.0007262446025235129
0.22509225092250923
0.0


In [23]:
# To do: Train 5 times and average the resulting metrics. 
# To do: 5 different random seeds as well. 
from sklearn.model_selection import KFold, RepeatedKFold
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score 
n_repeats = 5 # number of seeds 
n_splits = 10 # number of k splits 
precisions = []
recalls = [] 
f1s = [] 
accs = []
for _ in range(n_repeats): 
    kf = KFold(n_splits=n_splits,  shuffle=True) 
    precision = 0 
    recall = 0 
    f1 = 0 
    acc = 0 
    for i, (train, test) in enumerate(kf.split(mergeds)):  # split into k-folds
        print('split ',i)
        # extract train and test dataset for k-folds cross-validation
        train_ds = mergeds.loc[train] 
        test_ds = mergeds.loc[test] 
        # do zero-shot labelling on the test dataset 
        predicted_labels = test_ds['topics'].apply(Zero_Shot_sample)
        # compute metrics on the validation dataset 
        report = classification_report(test_ds['labels'].tolist(),predicted_labels.tolist()
                                       ,output_dict=True, zero_division=0)
        accuracy = accuracy_score(test_ds['labels'].tolist(), predicted_labels.tolist())
        # gather and normalize metrics by the length of the test dataset 
        acc += (accuracy * len(test_ds)) / len(mergeds)
        precision += (report['macro avg']['precision'] * len(test_ds)) / len(mergeds)
        recall += (report['macro avg']['recall'] * len(test_ds)) / len(mergeds)
        f1 += (report['macro avg']['f1-score'] * len(test_ds)) / len(mergeds)
    
    precisions.append(precision) 
    recalls.append(recall) 
    f1s.append(f1)
    accs.append(acc) 
# print the results
print('precision: ',np.mean(precisions))
print('precision std: ',np.std(precisions))
print('recall: ',np.mean(recalls))
print('recall std: ',np.std(recalls))
print('f1: ',np.mean(f1s))
print('f1 std: ',np.std(f1s))
print('accuracy: ',np.mean(accs))
print('accuracy std: ',np.std(accs))

split  0
split  1
split  2
split  3
split  4
split  5
split  6
split  7
split  8
split  9


ValueError: invalid literal for int() with base 10: ''